In [1]:
import os
import json
import time
import pandas as pd
from tqdm import tqdm
from groq import Groq
from dotenv import load_dotenv

In [2]:



# ============================================================
# 1. CONFIGURATION
# ============================================================

load_dotenv()

API_KEY = os.getenv("GROQ_API_KEY")

if not API_KEY:
    print("ERROR: GROQ_API_KEY not found.")
    print("Make sure your .env file contains:")
    print("GROQ_API_KEY=your-key-here")
    raise SystemExit


# Create Groq client
client = Groq(api_key=API_KEY)


# Groq model
MODEL_TO_USE = "openai/gpt-oss-20b"


# Input and output files
INPUT_FILE = "Product List - Miri product master product list.csv"
OUTPUT_FILE = "ProductList_master_cleaned.csv"

COLUMN_TO_CLEAN = "ProductName"

SAVE_CHUNK_SIZE = 50


# ============================================================
# 2. AI PROMPT
# ============================================================

SYSTEM_PROMPT = """
You are an expert data extraction and cleaning bot.

You will be given a single product string.

Your task is to extract:

1. main_product
2. product_type
3. SKU

Return ONLY valid JSON in exactly this format:

{
    "main_product": "",
    "product_type": "",
    "SKU": ""
}


RULES:

- main_product:
  The main product name, usually the first word(s).
  It can sometimes contain two or more words.
  It represents the brand, product line, or main identity.

- product_type:
  The middle/descriptive part of the product description.
  Examples:
  soap, Rose, Powder, Cutter, butter, masala, Chips, Biscuits.

- SKU:
  The measurement, quantity, count, or MRP if present.
  Examples:
  100gm, 1L, 5kg, RS 15, 15 RS, Rs.1, 10, 40w, pcs, ml,
  6PCS, MRP 105.

- Correct obvious spelling mistakes.

- Normalize casing.

- Remove unnecessary spaces.

- If a value cannot be found, return an empty string "".

- SINGLE WORD PRODUCTS:
  If the input is a single word such as:
  Egg
  Mushroom
  Paneer

  put it in main_product and keep:
  product_type = ""
  SKU = ""

- Find the SKU first.
  SKU usually contains numbers and is often near the end.

- Find the main_product.
  Usually the beginning of the product name.

- product_type is the meaningful descriptive portion
  between main_product and SKU.


EXAMPLES:


Input:
Doctor soap -100 gm

Output:
{
    "main_product": "Doctor Soap",
    "product_type": "Soap",
    "SKU": "100 gm"
}


Input:
Dyna Rose -50 gm

Output:
{
    "main_product": "Dyna",
    "product_type": "Rose",
    "SKU": "50 gm"
}


Input:
Honey And Turmeric Soap.

Output:
{
    "main_product": "Honey And Turmeric Soap",
    "product_type": "Soap",
    "SKU": ""
}


Input:
Spicy Chilli

Output:
{
    "main_product": "Spicy Chilli",
    "product_type": "",
    "SKU": ""
}


Input:
Egg

Output:
{
    "main_product": "Egg",
    "product_type": "",
    "SKU": ""
}


Input:
Paneer

Output:
{
    "main_product": "Paneer",
    "product_type": "",
    "SKU": ""
}


Input:
Banana Chips 50g

Output:
{
    "main_product": "Banana Chips",
    "product_type": "Chips",
    "SKU": "50g"
}


Input:
Charcoal Agarbatti

Output:
{
    "main_product": "Charcoal Agarbatti",
    "product_type": "Agarbatti",
    "SKU": ""
}


Input:
Oyester Mashroom

Output:
{
    "main_product": "Oyster Mushroom",
    "product_type": "",
    "SKU": ""
}


Input:
Cushion Cover

Output:
{
    "main_product": "Cushion Cover",
    "product_type": "Cover",
    "SKU": ""
}


Input:
Breeze Sandle Sparsh

Output:
{
    "main_product": "Breeze",
    "product_type": "Sandal Sparsh",
    "SKU": ""
}


Input:
Lays Classic 50g

Output:
{
    "main_product": "Lays",
    "product_type": "Classic",
    "SKU": "50g"
}


Input:
Parle-G Gold Biscuits -10 RS

Output:
{
    "main_product": "Parle-G",
    "product_type": "Gold Biscuits",
    "SKU": "10 RS"
}


Input:
unbranded loose sugar 1kg

Output:
{
    "main_product": "unbranded",
    "product_type": "loose sugar",
    "SKU": "1kg"
}


Input:
Gazak - Cutter

Output:
{
    "main_product": "Gazak",
    "product_type": "Cutter",
    "SKU": ""
}


Input:
Super Garam Masala Jeet - Rs.1

Output:
{
    "main_product": "Super Garam Masala",
    "product_type": "Jeet",
    "SKU": "Rs.1"
}


Input:
Phynail

Output:
{
    "main_product": "Phenyl",
    "product_type": "",
    "SKU": ""
}


Input:
Airtel - 10

Output:
{
    "main_product": "Airtel",
    "product_type": "",
    "SKU": "10"
}


Input:
P BINGO CHIPS MRP 105

Output:
{
    "main_product": "P BINGO",
    "product_type": "CHIPS",
    "SKU": "MRP 105"
}


Input:
CEAM BON(6PCS)RKS

Output:
{
    "main_product": "CEAM BON",
    "product_type": "",
    "SKU": "6PCS"
}


Input:
Flattened Rice

Output:
{
    "main_product": "Flattened Rice",
    "product_type": "Rice",
    "SKU": ""
}
"""


# ============================================================
# 3. CLEAN PRODUCT FUNCTION
# ============================================================

def clean_product_name(product_string):

    if not isinstance(product_string, str) or product_string.strip() == "":
        return {
            "main_product": "",
            "product_type": "",
            "SKU": ""
        }

    max_retries = 5

    for attempt in range(max_retries):

        try:

            response = client.chat.completions.create(
                model=MODEL_TO_USE,
                messages=[
                    {
                        "role": "system",
                        "content": SYSTEM_PROMPT
                    },
                    {
                        "role": "user",
                        "content": product_string
                    }
                ],
                temperature=0,
                response_format={
                    "type": "json_object"
                }
            )

            result_json = response.choices[0].message.content

            data = json.loads(result_json)

            return {
                "main_product": data.get("main_product", ""),
                "product_type": data.get("product_type", ""),
                "SKU": data.get("SKU", "")
            }


        except Exception as e:

            error_message = str(e)

            print("\n--- ERROR ---")
            print(f"Product: {product_string}")
            print(f"Attempt: {attempt + 1}/{max_retries}")
            print(f"Error: {error_message}")

            # Retry rate-limit errors
            if "429" in error_message:

                wait_time = 10 * (2 ** attempt)

                print(
                    f"Rate limit reached. "
                    f"Waiting {wait_time} seconds..."
                )

                time.sleep(wait_time)

                continue


            # Retry temporary server errors
            elif "503" in error_message:

                wait_time = 10 * (2 ** attempt)

                print(
                    f"Server temporarily unavailable. "
                    f"Waiting {wait_time} seconds..."
                )

                time.sleep(wait_time)

                continue


            else:

                print("Non-retryable error.")
                break


    print(
        f"\nFAILED after {max_retries} attempts: "
        f"{product_string}"
    )

    return {
        "main_product": "API_ERROR",
        "product_type": "Request failed",
        "SKU": product_string
    }


# ============================================================
# 4. MAIN FUNCTION
# ============================================================

def main():

    print("=" * 60)
    print("SCM PRODUCT LIST CLEANING USING GROQ")
    print("=" * 60)

    print(f"\nInput file: {INPUT_FILE}")
    print(f"Output file: {OUTPUT_FILE}")
    print(f"Model: {MODEL_TO_USE}")


    # --------------------------------------------------------
    # Load CSV
    # --------------------------------------------------------

    try:

        df = pd.read_csv(INPUT_FILE)

    except FileNotFoundError:

        print(
            f"\nERROR: Input file '{INPUT_FILE}' not found."
        )

        print(
            "Make sure the CSV is in the same folder."
        )

        return

    except Exception as e:

        print(f"\nERROR loading CSV: {e}")

        return


    print(f"\nLoaded {len(df)} total rows.")

    print("Available columns:")
    print(list(df.columns))


    # --------------------------------------------------------
    # Check column
    # --------------------------------------------------------

    if COLUMN_TO_CLEAN not in df.columns:

        print(
            f"\nERROR: Column '{COLUMN_TO_CLEAN}' "
            "was not found."
        )

        print("\nAvailable columns:")
        print(list(df.columns))

        return


    # --------------------------------------------------------
    # Output columns
    # --------------------------------------------------------

    new_columns = [
        "main_product",
        "product_type",
        "SKU"
    ]


    # --------------------------------------------------------
    # Resume check
    # --------------------------------------------------------

    start_row = 0

    if os.path.exists(OUTPUT_FILE):

        try:

            df_existing = pd.read_csv(OUTPUT_FILE)

            start_row = len(df_existing)

            print(
                f"\nExisting output file found."
            )

            print(
                f"Already processed rows: {start_row}"
            )

        except Exception:

            print(
                "\nCould not read existing output."
            )

            print("Starting from beginning.")


    # --------------------------------------------------------
    # Create output header
    # --------------------------------------------------------

    if start_row == 0:

        print("\nStarting from the beginning.")

        header_df = pd.DataFrame(
            columns=list(df.columns) + new_columns
        )

        header_df.to_csv(
            OUTPUT_FILE,
            index=False
        )


    # --------------------------------------------------------
    # Remaining data
    # --------------------------------------------------------

    df_to_process = df.iloc[start_row:]


    if len(df_to_process) == 0:

        print(
            "\nAll rows have already been processed."
        )

        return


    print(
        f"\nProcessing {len(df_to_process)} remaining rows..."
    )

    print("-" * 60)


    results_chunk = []


    # --------------------------------------------------------
    # Process products
    # --------------------------------------------------------

    for index, row in tqdm(
        df_to_process.iterrows(),
        total=len(df_to_process),
        desc="Cleaning products"
    ):

        original_name = row[COLUMN_TO_CLEAN]


        # AI cleaning
        cleaned_data = clean_product_name(
            original_name
        )


        # Small delay between requests
        time.sleep(2)


        # Create output row
        new_row_data = row.to_dict()

        new_row_data["main_product"] = (
            cleaned_data.get(
                "main_product",
                ""
            )
        )

        new_row_data["product_type"] = (
            cleaned_data.get(
                "product_type",
                ""
            )
        )

        new_row_data["SKU"] = (
            cleaned_data.get(
                "SKU",
                ""
            )
        )


        results_chunk.append(
            new_row_data
        )


        # ----------------------------------------------------
        # Save every 50 rows
        # ----------------------------------------------------

        if len(results_chunk) >= SAVE_CHUNK_SIZE:

            df_chunk = pd.DataFrame(
                results_chunk
            )

            df_chunk.to_csv(
                OUTPUT_FILE,
                mode="a",
                header=False,
                index=False
            )

            print(
                f"\nSaved {len(results_chunk)} rows."
            )

            results_chunk = []


    # --------------------------------------------------------
    # Save remaining rows
    # --------------------------------------------------------

    if len(results_chunk) > 0:

        df_chunk = pd.DataFrame(
            results_chunk
        )

        df_chunk.to_csv(
            OUTPUT_FILE,
            mode="a",
            header=False,
            index=False
        )

        print(
            f"\nSaved final {len(results_chunk)} rows."
        )


    # --------------------------------------------------------
    # Finished
    # --------------------------------------------------------

    print("\n" + "=" * 60)
    print("PROCESSING COMPLETE")
    print("=" * 60)

    print(
        f"\nCleaned data saved to:"
    )

    print(OUTPUT_FILE)


# ============================================================
# 5. RUN
# ============================================================

if __name__ == "__main__":
    main()

SCM PRODUCT LIST CLEANING USING GROQ

Input file: Product List - Miri product master product list.csv
Output file: ProductList_master_cleaned.csv
Model: openai/gpt-oss-20b

Loaded 50 total rows.
Available columns:
['ProductName']

Starting from the beginning.

Processing 50 remaining rows...
------------------------------------------------------------


Cleaning products:  48%|████▊     | 24/50 [03:08<05:27, 12.58s/it]


--- ERROR ---
Product: Tata Salt 1kg
Attempt: 1/5
Error: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01kxrrrx74ehmb7023eh4p54wy` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 5820, Requested 2953. Please try again in 5.7975s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Rate limit reached. Waiting 10 seconds...


Cleaning products:  60%|██████    | 30/50 [04:25<03:56, 11.82s/it]


--- ERROR ---
Product: Kissan Mixed Fruit Jam 500g
Attempt: 1/5
Error: Error code: 400 - {'error': {'message': "Failed to validate JSON. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'json_validate_failed', 'failed_generation': ''}}
Non-retryable error.

FAILED after 5 attempts: Kissan Mixed Fruit Jam 500g


Cleaning products:  78%|███████▊  | 39/50 [06:14<01:47,  9.81s/it]


--- ERROR ---
Product: Harpic Toilet Cleaner 500ml
Attempt: 1/5
Error: Error code: 400 - {'error': {'message': "Failed to validate JSON. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'json_validate_failed', 'failed_generation': ''}}
Non-retryable error.

FAILED after 5 attempts: Harpic Toilet Cleaner 500ml


Cleaning products: 100%|██████████| 50/50 [08:54<00:00, 10.69s/it]


Saved 50 rows.

PROCESSING COMPLETE

Cleaned data saved to:
ProductList_master_cleaned.csv
